In [ ]:
import bacco
import numpy as np
import h5py
import matplotlib.pyplot as plt
from scipy.spatial import KDTree

In [ ]:
import sys
sys.path.append("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils
import merger_tree_tools as mgt

In [ ]:
%load_ext autoreload
%autoreload 2

### Load the Zoom

In [ ]:
base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/LH_0/hydro_output/"

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

snap = 264

zoom_z0 = bacco.Simulation(
            basedir=base,
            halo_file="groups_{0:03d}/fof_subhalo_tab_{0:03d}".format(snap),
            sim_format='TNG500', fixedPk=True, use_orphans=False,
            tau=tau,
            ns=ns,
            sigma8=sigma8,
            use_ids=False,
            tree_file="groups_{0:03d}/subhalo_prog_{0:03d}".format(snap),
            dm_file="snapdir_{0:03d}/snapshot_{0:03d}".format(snap),
            numpart=4320,
        )

In [ ]:
xmatch = utils.cross_match(zoom_z0, snap=264, name='fiducial')

In [ ]:
firstsub = zoom_z0.fof['halo_firstsub'][xmatch['ind']]
nsubs    = zoom_z0.fof['halo_nsubs'][xmatch['ind']]

sub_indices = np.concatenate([np.arange(f, f + n) for f, n in zip(firstsub, nsubs)])

In [ ]:
snap_0 = 264
tree = mgt.tree(snap_0=snap_0, tree_format='zoom', name='fiducial', sim=zoom_z0) # class object containing methods related to the merger-trees
tree.read_tree_opt()

In [ ]:
# Load the Halo Selection
with open("/cosmos_storage/simulations/TNG_Family/MN5_resims/resims_info/hydro_halo_sel_1pmbin.txt") as f:
    final_sel = []

    for line in f.readlines():
        final_sel.append(int(line.split()[0]))

final_sel = np.array(final_sel)

# Load MTNG and get the fraction of halos to do the upweighting
mtng = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/TNG_Family/MTNG/", snap=264)

m200b = np.log10(1e10 * mtng.fof['halo_m200b'])

mbins = np.concatenate(
    (np.arange(11, 11.5, 0.0025),
    np.arange(11.5, 12.5, 0.005),
    np.arange(12.5, 13.5, 0.025),
    np.arange(13.5, 15.01, 0.125))
)

h_frac = np.zeros(len(final_sel))
for m in range(len(mbins)-1):
    h_frac[m] = np.where(( m200b[final_sel]>=mbins[m]) & ( m200b[final_sel]<mbins[m+1]))[0].shape[0] / \
             np.where(( m200b>=mbins[m]) & ( m200b<mbins[m+1]))[0].shape[0]

zoom_split = utils.split_halos(zoom_z0)

zoom_sel = {}

zoom_sel['sel'] = xmatch['ind'][:,np.newaxis,np.newaxis]
zoom_sel['h_frac'] = h_frac[np.newaxis, :]


In [ ]:
smf_z0 = zoom_split.halo_smf_draws(sel_mask=zoom_sel, nbins=15, draws=1, m_30kpc=True, depth=0, tree=tree, name='LH_0')

In [ ]:
smf_z1 = zoom_split.halo_smf_draws(sel_mask=zoom_sel, nbins=15, draws=1, m_30kpc=True, depth=264-179, tree=tree, name='LH_0')

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
ax.set_yscale('log')

ax.plot(smf_z0['mstar'][0], smf_z0['smf'][0], color="C0")
ax.plot(smf_z1['mstar'][0], smf_z1['smf'][0], color="C3")

### Let's load our z > 0 mass functions

In [ ]:
filebase = "/cosmos_storage/home/fgmaion/MTNG-resims/results/smf_high_z"

name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial']

Nbins_smf = 10

z_list = [0.25, 0.3, 0.5, 0.7, 0.99]
smf_z = {}

for n in range(len(z_list)):
    smf_z[z_list[n]] = {}
    
    for i in range(len(name_list)):
        smf_z[z_list[n]][name_list[i]] = np.load("{:s}/smf_{:s}_Nbins10_z{:.2f}.npy".format(filebase, name_list[i], z_list[n]), allow_pickle=True)

In [ ]:
Nbins_smf = 10

z_choice = 0.25 #0.3, 0.5, 0.7, 0.99
zoom_smf = {}

zoom_base = "/cosmos_storage/home/fgmaion/MTNG-resims/results/smf_high_z"

for i in range(len(name_list)):
    zoom_smf[name_list[i]] = np.load("{:s}/smf_{:s}_Nbins10_z{:.2f}.npy".format(zoom_base, name_list[i], z_choice), allow_pickle=True)[0]

In [ ]:
smf_z[0.25]['fiducial'][0]['mstar']

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

for n in range(len(z_list)):
    ax.plot( np.log10(smf_z[z_list[n]]['fiducial'][0]['mstar'][0]), np.log10(smf_z[z_list[n]]['fiducial'][0]['smf'][0]) )


In [ ]:
# fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

# for n in range(len(z_list)):
#     ax.plot( np.log10(smf_z[z_list[n]]['LH_22'][0]['mstar'][0]), np.log10(smf_z[z_list[n]]['LH_22'][0]['smf'][0]) )


### Let's check if this emulator works

In [ ]:
idx = tree.walk_tree()

sel_214 = np.zeros(len(final_sel), dtype=int)
sel_200 = np.zeros(len(final_sel), dtype=int)
sel_100 = np.zeros(len(final_sel), dtype=int)

for i in range(len(final_sel)):
    sel_214[i] = tree.group_nr[i][idx[i,snap_0-214]]
    sel_200[i] = tree.group_nr[i][idx[i,snap_0-200]]
    sel_100[i] = tree.group_nr[i][idx[i,snap_0-100]]

### Cross-Match Selected Halos

In general there is a large-scale shift between the positions of resimulated halos and the original ones. This is simply due to the poor resolution of large-scale forces

For this we can use the work done by Kurt Walsen, of finding the large-scale displacements between the zoomed regions and the original MTNG.

In the files <code>/lscratch/kwalsen/xmatch/*/selection_xmatch_deltas.csv</code> we have:

- Halo-index for z=0 MTNG group-files
- Halo-index for z=0 zoom group-files
- dx, dy, dz -- Large-Scale displacements used to find a good cross-match between full resimulated region

**I won't use this for now because I want this cross-matching at many different redshifts**


In [ ]:
# Even if I'm not using it now, here's how to read Kurt's data
fid_xm = utils.read_central_xmatch("fiducial")

### Now cross-match those at same redshift

In [ ]:
xm_214 = utils.cross_match(zoom_214, mtng_214, sel_214)

In [ ]:
ihalo = 400

In [ ]:
fig, ax = plt.subplots(1, 3, dpi=150, figsize=(16,5))

w = np.where(xm_214['dm']>0.3)

Xarr = np.log10(1e10*mtng_214.fof['halo_m200b'][sel_214])
Yarr = (zoom_214.fof['halo_m200b'][psel[xm_214['xm']]] / mtng_214.fof['halo_m200b'][sel_214])

Xarr_out = np.log10(1e10*mtng_214.fof['halo_m200b'][sel_214][w])
Yarr_out = (zoom_214.fof['halo_m200b'][psel[xm_214['xm']]][w] / mtng_214.fof['halo_m200b'][sel_214][w])

ax[0].plot(Xarr, Yarr, marker='o', color='C0', ms=3, ls='')
ax[0].plot(Xarr_out, Yarr_out, marker='o', color='C3', ms=3, ls='')

ax[0].plot(Xarr[ihalo], Yarr[ihalo], marker='*', color='y', ms=5, ls='')

ax[0].axhline(1, color='k')

ax[0].set_xlabel('Full MTNG M200')
ax[0].set_ylabel('Fiducial Zoom M200 / Full MTNG M200')

#

ax[1].set_xlabel('Full MTNG M200')
ax[1].set_ylabel('Distance$[Mpc/h]$')

ax[1].plot(np.log10(1e10*mtng_214.fof['halo_m200b'][sel_214]), xm_214['dm'], marker='o', color='C0', ms=3, ls='')
ax[1].plot(np.log10(1e10*mtng_214.fof['halo_m200b'][sel_214][ihalo]), xm_214['dm'][ihalo], marker='*', color='y', ms=5, ls='')

#ax[1].set_xlim(8,14)
ax[1].axhline(0.3, color='C3')

ax[1].set_yscale('log')

#

ax[2].set_xscale('log')

ax[2].set_xlabel('Full MTNG Vmax')
ax[2].set_ylabel('Zoom Vmax / Full MTNG Vmax')

ax[2].plot(np.log10(1e10*mtng_214.fof['halo_m200b'][sel_214]), mtng_214.sub['vmax'][mtng_214.fof['halo_firstsub'][sel_214]] / zoom_214.sub['vmax'][zoom_214.fof['halo_firstsub'][psel[xm_214['xm']]]], marker='o', color='C0', ms=3, ls='')
ax[2].plot(np.log10(1e10*mtng_214.fof['halo_m200b'][sel_214][w]), mtng_214.sub['vmax'][mtng_214.fof['halo_firstsub'][sel_214][w]] / zoom_214.sub['vmax'][zoom_214.fof['halo_firstsub'][psel[xm_214['xm']][w]]], marker='o', color='C3', ms=3, ls='')

ax[2].axhline(1, color='k')

### Let's work in a specific region

We will focus on one of the zoom halos

In [ ]:
Pzoom = zoom.fof['halo_pos'][psel][xmatch]
Pmtng = mtng.fof['halo_pos'][sel_z]

In [ ]:
center = Pmtng[ihalo]

In [ ]:
L_p = 5
z_slab = 5
xmin = np.array([center[0]-L_p/2,center[1]-L_p/2,center[2]-z_slab/2])

mtng_sel = np.where( (mtng.fof['halo_pos'][:,0]>(center[0]-L_p/2)) & (mtng.fof['halo_pos'][:,0]<(center[0]+L_p/2)) &\
                     (mtng.fof['halo_pos'][:,1]>(center[1]-L_p/2)) & (mtng.fof['halo_pos'][:,1]<(center[1]+L_p/2)) &\
                     (mtng.fof['halo_pos'][:,2]>(center[2]-z_slab/2)) & (mtng.fof['halo_pos'][:,2]<(center[2]+z_slab/2)) &\
                     (mtng.fof['halo_m200b']>0))[0]

In [ ]:
zoom_grid = {}

part = ['dm', 'gas', 'stars']

for type in range(len(part)):
    zoom_sel = np.where( (getattr(zoom_snap,part[type])['pos'][:,0]>(center[0]-L_p/2)) & (getattr(zoom_snap,part[type])['pos'][:,0]<(center[0]+L_p/2)) &\
                        (getattr(zoom_snap,part[type])['pos'][:,1]>(center[1]-L_p/2)) & (getattr(zoom_snap,part[type])['pos'][:,1]<(center[1]+L_p/2)) &\
                        (getattr(zoom_snap,part[type])['pos'][:,2]>(center[2]-z_slab/2)) & (getattr(zoom_snap,part[type])['pos'][:,2]<(center[2]+z_slab/2)) )[0]

    zoom_grid[part[type]] = bacco.statistics.compute_mesh(pos=getattr(zoom_snap, part[type])['pos'][zoom_sel] - xmin[np.newaxis,:], box=L_p, ngrid=512)

In [ ]:
sel_pos = (mtng.fof['halo_pos'][mtng_sel]-xmin[np.newaxis, :])/L_p * 512

In [ ]:
Pmtng_norm = (Pmtng-xmin[np.newaxis, :])/L_p * 512
Pzoom_norm = (Pzoom-xmin[np.newaxis, :])/L_p * 512

fig, ax = plt.subplots(3, 3, dpi=150, figsize=(15,15))

cmap = ['inferno', 'inferno', 'viridis']

for type in range(len(part)):
    ax[type, 0].imshow(np.log10(1+np.sum(zoom_grid[part[type]][0,:,:,:],axis=2)), cmap=cmap[type])
    ax[type, 1].imshow(np.log10(1+np.sum(zoom_grid[part[type]][0,:,:,:],axis=1)), cmap=cmap[type])
    ax[type, 2].imshow(np.log10(1+np.sum(zoom_grid[part[type]][0,:,:,:],axis=0)), cmap=cmap[type])

    for i in range(3):
        xticklabels = 1000 * ax[type, i].get_xticks() / 512 * L_p
        yticklabels = 1000 * ax[type, i].get_yticks() / 512 * L_p
        ax[type, i].set_xticklabels(['{:.0f}'.format(xticklabels[i]) for i in range(len(xticklabels))])
        ax[type, i].set_yticklabels(['{:.0f}'.format(yticklabels[i]) for i in range(len(yticklabels))])

    ax[type, 0].set_xlabel('y [kpc]', fontsize=14)
    ax[type, 0].set_ylabel('x [kpc]', fontsize=14)

    ax[type, 1].set_xlabel('z [kpc]', fontsize=14)
    ax[type, 1].set_ylabel('x [kpc]', fontsize=14)

    ax[type, 2].set_xlabel('z [kpc]', fontsize=14)
    ax[type, 2].set_ylabel('y [kpc]', fontsize=14)

    ax[type, 1].set_title(part[type], fontsize=14)

    for i in range(sel_pos.shape[0]):
        circle = plt.Circle((sel_pos[i,1],sel_pos[i,0]), mtng.fof['halo_r200c'][mtng_sel][i]/L_p*512, color='r', fill=False)
        ax[type, 0].add_artist(circle)
        circle = plt.Circle((sel_pos[i,2],sel_pos[i,0]), mtng.fof['halo_r200c'][mtng_sel][i]/L_p*512, color='r', fill=False)
        ax[type, 1].add_artist(circle)
        circle = plt.Circle((sel_pos[i,2],sel_pos[i,1]), mtng.fof['halo_r200c'][mtng_sel][i]/L_p*512, color='r', fill=False)
        ax[type, 2].add_artist(circle)

    ax[type, 0].plot( Pzoom_norm[ihalo][1], Pzoom_norm[ihalo][0], 'y*', ms=3)
    ax[type, 1].plot( Pzoom_norm[ihalo][2], Pzoom_norm[ihalo][0], 'y*', ms=3)
    ax[type, 2].plot( Pzoom_norm[ihalo][2], Pzoom_norm[ihalo][1], 'y*', ms=3)

    circle = plt.Circle((Pmtng_norm[ihalo][1], Pmtng_norm[ihalo][0]), mtng.fof['halo_r200c'][sel_z][ihalo]/L_p*512, color='y', fill=False)
    ax[type, 0].add_artist(circle)
    circle = plt.Circle((Pmtng_norm[ihalo][2], Pmtng_norm[ihalo][0]), mtng.fof['halo_r200c'][sel_z][ihalo]/L_p*512, color='y', fill=False)
    ax[type, 1].add_artist(circle)
    circle = plt.Circle((Pmtng_norm[ihalo][2], Pmtng_norm[ihalo][1]), mtng.fof['halo_r200c'][sel_z][ihalo]/L_p*512, color='y', fill=False)
    ax[type, 2].add_artist(circle)

plt.savefig('../images/zoom_snap{:d}_{:d}_fid.pdf'.format(snap, ihalo), bbox_inches='tight')

### Save the cross-match

In [ ]:
np.save("/lscratch/fgmaion/MTNG-resims/cross-match/xmatch_fid.npy", [{'xmatch':psel[xmatch], 'dmatch':dmatch}])

In [ ]:
import h5py

In [ ]:
with h5py.File('/cosmos_storage/data_sharing/MN5_resims/fiducial/hydro_output/snapdir_094/snapshot_094.0.hdf5', 'r') as f:
    print(np.uint64(f['PartType1']['ParticleIDs']))

In [ ]:
ic_file = '/cosmos_storage/data_sharing/MN5_resims/ICs_1pmbin_2160/output/snapdir_000/snapshot_ics_000'

ptype=1
part_type = 'PartType{0}'.format(ptype)

with h5py.File('/cosmos_storage/data_sharing/MN5_resims/ICs_1pmbin_2160/output/snapdir_000/snapshot_ics_000.1.hdf5', 'r') as f:
    nfiles = f['Header'].attrs['NumFilesPerSnapshot']
    numpart = f['Header'].attrs['NumPart_Total'][ptype]



In [ ]:
ids_ICs = np.zeros((numpart), dtype=np.int64)

istart = np.uint64(0)
for ifile in range(0, nfiles):

    ffname = ic_file+'.hdf5' if nfiles== 1 else ic_file+(
        '.%d.hdf5'%ifile)

    with h5py.File(ffname, 'r') as f: 
        npts = np.uint64(f['Header'].attrs['NumPart_ThisFile'][ptype])

        if npts > 0:
            ids_ICs[istart:istart + npts] = np.int64(f[part_type]['ParticleIDs'])

        istart += npts

In [ ]:
file = '/cosmos_storage/data_sharing/MN5_resims/fiducial/hydro_output/snapdir_264/snapshot_264'

ptype=1
part_type = 'PartType{0}'.format(ptype)

with h5py.File('/cosmos_storage/data_sharing/MN5_resims/fiducial/hydro_output/snapdir_264/snapshot_264.0.hdf5', 'r') as f:
    nfiles = f['Header'].attrs['NumFilesPerSnapshot']
    numpart = f['Header'].attrs['NumPart_Total'][ptype]


In [ ]:
ids = np.zeros((numpart), dtype=np.int64)

istart = np.uint64(0)
for ifile in range(0, nfiles):

    ffname = file+'.hdf5' if nfiles== 1 else file+(
        '.%d.hdf5'%ifile)

    with h5py.File(ffname, 'r') as f: 
        npts = np.uint64(f['Header'].attrs['NumPart_ThisFile'][ptype])

        if npts > 0:
            ids[istart:istart + npts] = np.int64(f[part_type]['ParticleIDs'])

        istart += npts

In [ ]:
sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

base_ics = "/cosmos_storage/data_sharing/MN5_resims/ICs_1pmbin_2160/output/"
zoom_ics = bacco.Simulation(basedir=base_ics, halo_file="groups_000/fof_subhalo_tab_000", dm_file="snapdir_000/snapshot_ics_000", sim_format='TNG500', fixedPk=True, use_orphans=False,\
                        tau=tau, ns=ns, sigma8=sigma8, use_ids=True, numpart=4320**3)

In [ ]:
zoom_ics.dm['ids']

In [ ]:
with h5py.File("/cosmos_storage/simulations/TNG_Family/MN5_resims/fiducial/hydro_output/treedata/trees.0.hdf5", 'r') as f:
    print(f.keys())
    print(f['TreeHalos'].keys())